# [7.5] Predictive Concept Decoders - Solutions

This notebook follows the same path as the exercise notebook: validate local helper contracts, inspect the CPU smoke report, inspect the committed CUDA signature result, then optionally run the live CUDA preflight through `solutions.py`.

<details>
<summary>Expected output</summary>

All local tests should pass. The signature result should show `pcd_accuracy = 1.0`, baselines at or below `0.5`, question-shuffle accuracy at `0.0`, stable top concepts, and top-removal delta larger than the low-margin active-control delta.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part5_predictive_concept_decoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_predictive_concept_decoders.tests as tests

GT_TIER = "GT-3"
EXERCISE_ID = "7_5_predictive_concept_decoders"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1 minute for the CUDA preflight"
REQUIRES_GPU = True

from chapter7_activation_to_language.exercises.part5_predictive_concept_decoders import solutions


## Unit Contracts

Each test sits close to the learner function it validates. Small tests catch shape, alignment, sparsity, and interaction mistakes before the real model path.

<details>
<summary>Help - why so many tests?</summary>

The live `gelu-1l` report is too coarse for debugging. If the final PCD fails, you want to know whether the bug is batch alignment, sparse concept encoding, interaction features, training, comparison, or audit controls.

</details>


In [ ]:
tests.test_build_pcd_question_batch_validates_shapes_and_questions(
    solutions.build_pcd_question_batch,
    solutions.default_pcd_questions,
)
tests.test_build_pcd_question_batch_rejects_empty_and_bad_question_ids(
    solutions.build_pcd_question_batch,
)
tests.test_sparse_concept_encode_and_sparsity_controls(
    solutions.sparse_concept_encode,
    solutions.concept_sparsity_report,
)
tests.test_sparse_concept_encode_and_sparsity_reject_bad_controls(
    solutions.sparse_concept_encode,
    solutions.concept_sparsity_report,
)
tests.test_question_conditioned_decoder_uses_question_information(
    solutions.question_conditioned_decoder_logits,
)
tests.test_question_conditioned_decoder_respects_arbitrary_weight_and_bias(
    solutions.question_conditioned_decoder_logits,
)
tests.test_question_conditioned_features_and_decoder_reject_bad_shapes(
    solutions.question_conditioned_concept_features,
    solutions.question_conditioned_decoder_logits,
)
tests.test_trained_question_conditioned_decoder_learns_concept_question_interaction(
    solutions.question_conditioned_concept_features,
    solutions.train_question_conditioned_decoder,
    solutions.question_conditioned_decoder_logits,
)
tests.test_train_question_conditioned_decoder_rejects_empty_and_misaligned_batches(
    solutions.train_question_conditioned_decoder,
)
tests.test_pcd_comparison_report_beats_baselines(solutions.pcd_comparison_report)
tests.test_pcd_comparison_report_scores_each_baseline_independently(
    solutions.pcd_comparison_report,
)
tests.test_concept_stability_removal_and_audit_controls(
    solutions.concept_stability_report,
    solutions.concept_removal_report,
    solutions.concept_audit_report,
)
tests.test_concept_audit_controls_reject_bad_inputs(
    solutions.concept_stability_report,
    solutions.concept_removal_report,
    solutions.concept_audit_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## CPU Contract

The smoke report should already show the shape of the section: sparse concepts, trained interaction decoder, baseline comparison, seed stability, low-margin active-control removal, and concept-name audit.

<details>
<summary>Expected output</summary>

The trained toy decoder should report `conditioned_shape = [4, 4]`, `train_accuracy = 1.0`, and predictions `[1, 0, 0, 1]`.

</details>

<details>
<summary>Common bug</summary>

If the trained predictions are correct but `conditioned_shape` is `[4, 2]`, you trained a probe, not the question-conditioned concept decoder.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["sparse_encoding"]["sparsity"]["passes_sparsity"]
assert contract["decoder"] == [[3.0, 0.0], [0.0, 3.0]]
assert contract["decoder_training"]["conditioned_shape"] == [4, 4]
assert contract["decoder_training"]["train_accuracy"] == 1.0
assert contract["decoder_training"]["predictions"] == contract["decoder_training"]["answer_ids"]
assert contract["comparison"]["pcd_accuracy"] == 1.0
assert contract["comparison"]["probe_accuracy"] == 0.5
assert contract["comparison"]["beats_best_baseline"]
assert contract["stability"]["stable"]
assert contract["removal"]["random_removal_does_less"]
assert contract["audit"]["names_expected_cluster"]
contract


## Signature Result

Now inspect the accepted CUDA report. The important result is the pattern of controls, not only the final boolean.

<details>
<summary>Interpreting the signature result</summary>

The PCD is perfect on the held-out question rows, while the best baseline stays at chance in the accepted run. Shuffling the questions collapses accuracy, which shows that the question alignment matters. Top-removal changing the answer more than the low-margin active control makes the selected concepts useful to the decoder.

</details>

<details>
<summary>Common bug</summary>

Do not treat `selected_concept_names` as semantic proof. In this preflight, names come from constructed directions and are only audit handles.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"] and report["tests_passed"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["activation_shape"] == [8, 512]
assert gpu["concept_shape"] == [8, 8]
assert gpu["conditioned_concept_shape"] == [32, 32]
assert gpu["question_count"] == 4
assert gpu["pcd_accuracy"] == 1.0
assert gpu["probe_accuracy"] == 0.5
assert gpu["best_baseline_accuracy"] <= 0.75
assert gpu["beats_probe"] and gpu["beats_best_baseline"]
assert gpu["pcd_decoder_train_accuracy"] == 1.0
assert gpu["question_shuffle_accuracy"] <= 0.75
assert gpu["pcd_seed_min_accuracy"] == 1.0
assert gpu["mean_pairwise_jaccard"] == 1.0
assert gpu["stable"]
assert gpu["top_removal_changed"]
assert gpu["random_removal_does_less"]
assert gpu["random_removed_concept_active"]
assert gpu["names_expected_cluster"]
assert gpu["within_vram_budget"]

labels = ["PCD", "probe", "best baseline", "shuffled"]
values = [
    gpu["pcd_accuracy"],
    gpu["probe_accuracy"],
    gpu["best_baseline_accuracy"],
    gpu["question_shuffle_accuracy"],
]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(labels, values, color=["#2563eb", "#64748b", "#64748b", "#f97316"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("accuracy")
ax.set_title("7.5 PCD signature result")
ax.grid(axis="y", alpha=0.25)
plt.show()

{
    "pcd_accuracy": gpu["pcd_accuracy"],
    "probe_accuracy": gpu["probe_accuracy"],
    "best_baseline_accuracy": gpu["best_baseline_accuracy"],
    "question_shuffle_accuracy": gpu["question_shuffle_accuracy"],
    "selected_concept_names": gpu["selected_concept_names"],
    "top_vs_control_delta": (
        round(gpu["top_removal_delta"], 4),
        round(gpu["random_removal_delta"], 4),
    ),
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}


## Live CUDA Path

The report above is the committed review artifact. This cell runs the live CUDA preflight through `solutions.py` on the current machine.

<details>
<summary>Expected output</summary>

`preflight_passed` should be `True`, peak VRAM should stay below the 24GB budget, and the metric pattern should match the committed report qualitatively.

</details>


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_full_experiment(max_vram_gb=max_vram_gb)


live_gpu = run_full_experiment(max_vram_gb=24.0)
assert live_gpu["preflight_passed"]
assert live_gpu["peak_vram_gb"] <= 24.0
assert live_gpu["pcd_accuracy"] == 1.0
assert live_gpu["beats_best_baseline"]
assert live_gpu["question_shuffle_accuracy"] <= 0.75
assert live_gpu["top_removal_changed"]
assert live_gpu["random_removal_does_less"]
{
    "preflight_passed": live_gpu["preflight_passed"],
    "pcd_accuracy": live_gpu["pcd_accuracy"],
    "probe_accuracy": live_gpu["probe_accuracy"],
    "question_shuffle_accuracy": live_gpu["question_shuffle_accuracy"],
    "peak_vram_gb": round(live_gpu["peak_vram_gb"], 3),
}


## Limitations

This is a GT-3 local mini-PCD preflight on one pinned `gelu-1l` hook and tiny safe prompt splits. It is not a broad PCD benchmark, not model-level causal proof, not semantic proof from names, and not a full Activation Oracle comparison.

## Further Research

Scale the question bank, replace constructed concept directions with learned dictionaries or SAEs, run true model-level ablations, add more layers and seeds, and keep the same baseline/shuffle/removal controls as the task grows.
